# 12 · Barrido de entrenamiento

Entrena una versión del modelo downstream por cada receta de la rejilla y por cada tarea, cacheando las métricas en CSV tras cada receta. Es **reanudable**: al relanzarlo salta todo lo que ya está en el CSV.

**Entradas**

- `data/processed/ventanas.npz`
- `data/synthetic/*.npz`
- `results/metricas/barrido.csv (si existe)`

**Salidas**

- `results/metricas/barrido.csv`
- `results/historiales/*.csv`

**Tiempo estimado:** ~5 h en CPU la primera vez. Reanudable: puede ejecutarse en varias sesiones y repartirse entre máquinas fusionando después los CSV.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import ventanas

part = ventanas.cargar_procesado()
train, val, test = part.train, part.val, part.test
print(train, val, test, sep="\n")

In [ ]:
from tqdm.auto import tqdm

from src import downstream, evaluacion, mezclas
from src.generadores.base import REGISTRO, GeneradorSintetico

v = config.ventanas()
n_regimenes = config.n_regimenes()
forma_entrada = train.X.shape[1:]

# La arquitectura y el presupuesto se LEEN del fichero que congeló el cuaderno 03,
# no se heredan del valor por defecto del módulo. Si alguien edita `downstream.py`
# entre los dos cuadernos, aquí se detecta en vez de contaminar en silencio los
# cientos de entrenamientos del barrido (D20).
arquitectura, presupuesto = downstream.cargar_congelada()

disponibles = [n for n in sorted(REGISTRO) if (src.DIR_SINTETICO / (n + ".npz")).exists()]
bancos = {n: GeneradorSintetico.importar_muestras(n) for n in disponibles}

recetas = mezclas.rejilla(disponibles)
print("Congelada:", arquitectura, "· huella", downstream.huella(arquitectura, presupuesto))
print(presupuesto)
print(len(disponibles), "generadores ·", len(recetas), "recetas ·",
      len(recetas) * len(downstream.TAREAS), "entrenamientos")

## Por qué tiene que ser reanudable

Son varios cientos de entrenamientos en CPU. Un kernel que se cae a las tres horas,
un portátil que se suspende o un reparto del trabajo entre los tres integrantes
convierten la reanudación en un requisito, no en una comodidad.

El mecanismo es deliberadamente simple: la clave de una fila es
`(identificador, tarea)`, se lee el CSV existente al arrancar y se salta cualquier
combinación que ya esté. El CSV se reescribe entero tras **cada receta**, de modo
que una interrupción cuesta como mucho un entrenamiento.

Consecuencia práctica: para rehacer un experimento hay que borrar su fila del CSV, o
el barrido lo dará por hecho.

In [ ]:
NOMBRE_TABLA = "barrido"
ruta_csv = src.DIR_METRICAS / (NOMBRE_TABLA + ".csv")

filas = pd.read_csv(ruta_csv).to_dict("records") if ruta_csv.exists() else []
hechas = {(f["identificador"], f["tarea"]) for f in filas}

print("Filas ya cacheadas:", len(filas))
print("Pendientes:", len(recetas) * len(downstream.TAREAS) - len(hechas))

## El bucle

Cada receta monta su conjunto de entrenamiento una sola vez y lo reutiliza para las
dos tareas: el montaje incluye un muestreo del banco sintético y repetirlo daría
conjuntos distintos por tarea, lo que impediría comparar régimen y volatilidad sobre
la misma versión del dataset.

Validación y test son siempre reales. La parada temprana restaura los pesos de la
mejor época de validación; sin ella los datasets con muchos sintéticos sobreajustan
y la comparación mediría cuánto sobreajusta cada generador, no cuánto aporta.

Se guarda el historial de cada entrenamiento: el enunciado exige poder mostrar las
curvas de loss de cada modelo, y con este volumen reentrenar para rehacer una figura
no es una opción.

In [ ]:
def objetivo(conjunto, tarea):
    return conjunto.y_reg if tarea == "regimen" else conjunto.y_vol


for receta in tqdm(recetas, desc="recetas"):
    pendientes = [t for t in downstream.TAREAS if (receta.identificador, t) not in hechas]
    if not pendientes:
        continue

    conjunto = mezclas.montar(
        receta, train, bancos.get(receta.generador), v, n_regimenes
    )
    n_reales = len(train) if receta.n_reales is None else receta.n_reales

    for tarea in pendientes:
        modelo = downstream.construir(
            tarea, forma_entrada, n_clases=n_regimenes, arquitectura=arquitectura
        )
        historial = downstream.entrenar(
            modelo,
            conjunto.X, objetivo(conjunto, tarea),
            val.X, objetivo(val, tarea),
            tarea=tarea, presupuesto=presupuesto, verboso=0,
        )
        downstream.guardar_historial(historial, receta.identificador + "__" + tarea)

        metricas = evaluacion.evaluar(modelo, test.X, objetivo(test, tarea), tarea)
        filas.append({
            "identificador": receta.identificador,
            "generador": receta.generador,
            "n_reales": "todos" if receta.n_reales is None else str(receta.n_reales),
            "n_reales_num": n_reales,
            "ratio": receta.ratio,
            "politica": receta.politica,
            "tarea": tarea,
            "n_entrenamiento": len(conjunto),
            "epocas": len(historial.history["loss"]),
            **metricas,
        })
        hechas.add((receta.identificador, tarea))

    evaluacion.acumular(filas, NOMBRE_TABLA)

print("Barrido completo:", len(filas), "filas.")

## Control de integridad

Antes de dar el barrido por bueno hay que comprobar que no falta ninguna
combinación y que ninguna métrica salió indefinida. Un `NaN` en `recall_crisis`
significa que el test no contenía ninguna ventana de crisis, y entonces toda la
lectura del informe cambia.

Las dos tareas escriben columnas distintas en la misma tabla, así que los huecos
entre columnas de tareas ajenas son esperados: el contraste se hace tarea a
tarea.

In [ ]:
tabla = evaluacion.cargar_metricas(NOMBRE_TABLA)

esperadas = {(r.identificador, t) for r in recetas for t in downstream.TAREAS}
obtenidas = set(zip(tabla["identificador"], tabla["tarea"]))

print("Filas:", len(tabla), "· esperadas:", len(esperadas))
print("Faltan:", sorted(esperadas - obtenidas)[:5] or "ninguna")
print("Duplicadas:", int(tabla.duplicated(["identificador", "tarea"]).sum()))

CLAVES = {"regimen": ["f1_macro", "balanced_accuracy", "recall_crisis"],
          "volatilidad": ["mae", "qlike", "r2"]}

for tarea, columnas in CLAVES.items():
    subconjunto = tabla.loc[tabla["tarea"] == tarea, columnas]
    print(tarea, "· métricas indefinidas:", int(subconjunto.isna().sum().sum()))

In [ ]:
tabla.groupby(["tarea", "generador"]).size().rename("entrenamientos").to_frame()

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_METRICAS / "barrido.csv",
    src.DIR_HISTORIALES,
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
